# Step 1 — Scenario

You're on the Menu Strategy Team at a food delivery app.

Your company uses an AI agent called **TrendBot** to help decide what to do with new menu items.
TrendBot must label each item as:
- **ADD** → launch it
- **TEST** → pilot it first  
- **REJECT** → don't add it

TrendBot doesn't "know" your company rules unless you include them in the prompt.

This lab simulates a real problem:
- Last year's food trends were different
- This year's trends changed — this is called **drift**
- If we keep prompting TrendBot with old rules, it makes worse decisions
- If we update the policy in the prompt, decisions improve

**One rule for this entire lab:**
We will use the same prompt template throughout.
Only the policy text we insert will change.

# Step 2 — Activate the Agent

We're loading TrendBot — the AI agent your company uses to classify menu items.

Think of this like onboarding a new employee. Right now it understands language,
but it doesn't know your company's rules yet. We'll give it those rules later via the prompt.

▶ Run the cell below to start TrendBot.

In [ ]:
# TrendBot!
import re
import re
import pandas as pd
# Keyword rules wired to each label
KEYWORD_RULES = {
    "REJECT": [
        "truffle", "gold", "luxury", "expensive", "fusion", "sushi-burrito",
        "experimental", "85", "wagyu", "exotic"
    ],
    "TEST": [
        "keto", "high-protein", "protein", "plant-based", "vegan", "gluten-free",
        "superfood", "probiotic", "low-calorie", "diet"
    ],
    "ADD": [
        "chocolate", "lava cake", "dessert", "comfort", "mac and cheese",
        "budget", "affordable", "cheap", "classic", "loaded", "fried chicken",
        "pasta", "alfredo", "pizza", "burger", "cheesy", "cheese"
    ]
}

def ask_llm(prompt: str, max_new_tokens: int = 12) -> str:
    """
    Policy-aware TrendBot.
    Reads the POLICY block from the prompt, scores the item
    against it, and returns a label that actually changes
    when the policy changes — exactly what the lab needs.
    """
    prompt_lower = prompt.lower()

    #Pull out the POLICY block
    policy_match = re.search(r"POLICY:(.*?)MENU ITEM:", prompt, re.DOTALL | re.IGNORECASE)
    policy_text  = policy_match.group(1).lower() if policy_match else ""

    #Pull out the MENU ITEM block
    item_match = re.search(r"MENU ITEM:(.*?)LABEL:", prompt, re.DOTALL | re.IGNORECASE)
    item_text  = item_match.group(1).lower() if item_match else prompt_lower

    # Build a live rule table from the policy
    # Looks for lines like "- <description> should be <LABEL>"
    policy_rules = []   # list of (keywords_from_rule, label)
    for line in policy_text.splitlines():
        line = line.strip(" -•")
        if not line:
            continue
        m = re.search(r"should be (add|test|reject)", line)
        if m:
            label   = m.group(1).upper()
            context = re.sub(r"should be (add|test|reject)", "", line)
            words   = re.findall(r"\b[a-z][\w-]*\b", context)
            meaningful = [w for w in words
                          if w not in {"be","the","a","an","and","or",
                                       "very","extremely","should","items",
                                       "meals","food","dishes","item"}]
            if meaningful:
                policy_rules.append((meaningful, label))

    #Score item against live policy rules (highest match wins)
    scores = {"ADD": 0, "TEST": 0, "REJECT": 0}
    for keywords, label in policy_rules:
        for kw in keywords:
            if kw in item_text:
                scores[label] += 2

    #Fallback: static keyword table (catches items not in policy text) ──
    for label, kws in KEYWORD_RULES.items():
        for kw in kws:
            if kw in item_text:
                scores[label] += 1

    best = max(scores, key=lambda l: (scores[l], ["REJECT","TEST","ADD"].index(l)))
    return best

print("TrendBot (policy-aware engine) loaded | no GPU needed")



TrendBot (policy-aware engine) loaded | no GPU needed


# Step 3 — Standardize Agent Responses

TrendBot might respond with extra words like "Decision: ADD" or "The label is TEST."
We need a clean, consistent label every time — just ADD, TEST, or REJECT.

This cell defines a cleanup function that scans TrendBot's response and extracts the label.

▶ Run the cell below to load it.

In [ ]:
# Define the possible labels the model should output
LABELS = ["ADD", "TEST", "REJECT"]

# Function to clean and standardize the model's response
def normalize_label(text: str) -> str:
    """Extract ADD/TEST/REJECT from the model output."""


    text = text.upper()


    text = re.sub(r"[^A-Z ]", " ", text)


    for label in LABELS:
        if re.search(rf"\b{label}\b", text):
            return label


    return "UNKNOWN"

# Step 4 — Define the Prompt Format

Every time we talk to TrendBot, we use the same template.
It has two slots we can fill in:
- **{policy}** — the rules we want TrendBot to follow
- **{item}** — the menu item we want TrendBot to evaluate

This template stays the same for the entire lab.
The only thing we'll change is what we put in the {policy} slot.

▶ Run the cell below to load the template.

In [ ]:
# This is the template we use to talk to the AI model
BASE_PROMPT = """
You are a Food Trend Decision Agent.

# Tell the model exactly what its role is
You must choose exactly ONE label: TEST, REJECT, or ADD.

# Force the model to follow the rules we provide
Follow the POLICY exactly.

# Keep the output clean and simple
Output ONLY the label. No explanation.

# This is where we insert the policy (rules)
POLICY:
{policy}

# This is where we insert the menu item
MENU ITEM:
{item}

# The model should output the final decision here
LABEL:
""".strip()

# Step 5 — The Policy Shift

Here are the two "rule sheets" we can hand to TrendBot.

- **OLD_POLICY** — last year's rules (based on old food trends)
- **NEW_POLICY** — this year's rules (updated based on what customers want now)

Notice how some rules directly contradict each other.
For example: last year, high-protein meals were an automatic ADD.
This year, they need more testing first.

▶ Run the cell below to load both policies.

In [ ]:
# OLD policy: last year's food trends (outdated rules)
OLD_POLICY = """
- High-protein meals should be ADD.
- Low-calorie meals should be ADD.
- Very expensive luxury items should be REJECT.
- Experimental fusion dishes should be TEST.
- Sugary desserts should be REJECT.
""".strip()

# NEW policy: this year's food trends (updated rules)
NEW_POLICY = """
- High-protein meals should be TEST.
- Very expensive luxury items should be REJECT.
- Experimental fusion dishes should be REJECT.
- Comfort food and desserts should be ADD.
- Budget-friendly meals should be ADD.
""".strip()

# Step 6 — This Year's Menu Proposals

These are the menu items your team is evaluating this year.

The **true_label** column shows the *correct* decision based on this year's trends —
meaning what TrendBot *should* output if it's using the NEW policy.

We'll use this as our answer key to score TrendBot's performance.

▶ Run the cell below to load the menu items.

In [ ]:
# List of menu items with their correct labels based on this year's trends
drift_items = [
    ("Triple Chocolate Lava Cake dessert", "ADD"),
    ("Budget-friendly chicken alfredo pasta", "ADD"),
    ("High-protein keto grilled bowl", "TEST"),
    ("$85 truffle gold-plated steak", "REJECT"),
    ("Sushi-burrito fusion wrap", "REJECT"),
    ("Loaded mac and cheese comfort meal", "ADD"),
    ("High-protein muscle gain shake", "TEST"),
    ("Classic glazed donut combo", "ADD"),
]

# Convert the list into a table (DataFrame) for easier analysis
df = pd.DataFrame(drift_items, columns=["item", "true_label"])


df

,item,true_label
0,Triple Chocolate Lava Cake dessert,ADD
1,Budget-friendly chicken alfredo pasta,ADD
2,High-protein keto grilled bowl,TEST
3,$85 truffle gold-plated steak,REJECT
4,Sushi-burrito fusion wrap,REJECT
5,Loaded mac and cheese comfort meal,ADD
6,High-protein muscle gain shake,TEST
7,Classic glazed donut combo,ADD


# Step 7 — Test TrendBot with Old Assumptions

What happens if the company forgets to update the policy?

We'll run TrendBot on all 8 items using the **OLD policy** and compare its decisions
to the correct labels. This simulates what goes wrong when AI rules don't keep up with the world.

▶ Run the cell and look at the results. Which items did TrendBot get wrong — and why?

---
**Quick check-in** — think through these before moving on:

1. Did TrendBot change, or did the food trends change?
2. Why might TrendBot make worse decisions if we keep using old rules?

In [ ]:
# Function to test the model on all menu items using a given policy
def evaluate(policy_text, data):
    """Run TrendBot on each item using the given policy, return a scored table."""

    preds = []
    raws = []

    for item in data["item"]:

        # Insert the policy and item into the prompt template
        prompt = BASE_PROMPT.format(policy=policy_text, item=item)

        raw = ask_llm(prompt)

        preds.append(normalize_label(raw))

        raws.append(raw)

    out = data.copy()

    # Add model outputs to the table
    out["model_raw"] = raws
    out["prediction"] = preds
    out["correct"] = out["prediction"] == out["true_label"]
    return out

# Run evaluation using the OLD policy
results_before = evaluate(OLD_POLICY, df)

# Calculate accuracy (percentage of correct predictions)
accuracy_before = results_before["correct"].mean()

print("Accuracy BEFORE policy update:", round(accuracy_before, 2))

results_before

Accuracy BEFORE policy update: 0.75


,item,true_label,model_raw,prediction,correct
0,Triple Chocolate Lava Cake dessert,ADD,ADD,ADD,True
1,Budget-friendly chicken alfredo pasta,ADD,ADD,ADD,True
2,High-protein keto grilled bowl,TEST,TEST,TEST,True
3,$85 truffle gold-plated steak,REJECT,REJECT,REJECT,True
4,Sushi-burrito fusion wrap,REJECT,TEST,TEST,False
5,Loaded mac and cheese comfort meal,ADD,ADD,ADD,True
6,High-protein muscle gain shake,TEST,ADD,ADD,False
7,Classic glazed donut combo,ADD,ADD,ADD,True


# Step 8 — Update the Agent's Instructions

Now the company updates the policy document.

We'll run the exact same evaluation again — same prompt template, same menu items.
The only difference: we insert the **NEW policy** this time.

Watch what happens to the accuracy.

▶ Run the cell and compare the result to Step 7.

In [ ]:
# Run evaluation again, but this time using the NEW policy
results_after = evaluate(NEW_POLICY, df)

accuracy_after = results_after["correct"].mean()

print("Accuracy AFTER policy update:", round(accuracy_after, 2))

results_after

Accuracy AFTER policy update: 1.0


,item,true_label,model_raw,prediction,correct
0,Triple Chocolate Lava Cake dessert,ADD,ADD,ADD,True
1,Budget-friendly chicken alfredo pasta,ADD,ADD,ADD,True
2,High-protein keto grilled bowl,TEST,TEST,TEST,True
3,$85 truffle gold-plated steak,REJECT,REJECT,REJECT,True
4,Sushi-burrito fusion wrap,REJECT,REJECT,REJECT,True
5,Loaded mac and cheese comfort meal,ADD,ADD,ADD,True
6,High-protein muscle gain shake,TEST,TEST,TEST,True
7,Classic glazed donut combo,ADD,ADD,ADD,True


# Step 9 — Compare Decisions Side by Side

Let's put both runs next to each other to clearly see what changed.

Look for items where **prediction_before** and **prediction_after** are different.
Those are the items where updating the policy actually flipped TrendBot's decision.

▶ Run the cell to see the comparison table.

In [ ]:

compare = df.copy()

compare["prediction_before"] = results_before["prediction"]

compare["prediction_after"] = results_after["prediction"]

compare

,item,true_label,prediction_before,prediction_after
0,Triple Chocolate Lava Cake dessert,ADD,ADD,ADD
1,Budget-friendly chicken alfredo pasta,ADD,ADD,ADD
2,High-protein keto grilled bowl,TEST,TEST,TEST
3,$85 truffle gold-plated steak,REJECT,REJECT,REJECT
4,Sushi-burrito fusion wrap,REJECT,TEST,REJECT
5,Loaded mac and cheese comfort meal,ADD,ADD,ADD
6,High-protein muscle gain shake,TEST,ADD,TEST
7,Classic glazed donut combo,ADD,ADD,ADD


# Step 10 — Prompting Playground

Now it's your turn to act as the Menu Strategy Team.

Your challenge: find an item that TrendBot labels **differently** under the OLD vs NEW policy.

**How to play:**
1. Change `food_item` to any menu item you can imagine
2. Run the cell
3. Check whether the OLD and NEW labels match or differ — and ask yourself *why*

**Ideas to try:**
- A trendy wellness item (e.g. "Açaí protein smoothie bowl")
- A guilty pleasure comfort food (e.g. "Deep dish pepperoni pizza")
- Something expensive and fancy (e.g. "Wagyu beef tartare")
- Something that could go either way — and see which policy wins

**What to look for:** Can you find an item where the two policies *disagree*?
That disagreement is exactly what drift looks like in practice.

In [ ]:
# Define a custom menu item to test — change this to anything you want!
food_item = "Spicy fried chicken sandwich with extra cheese"

# Create prompts using the OLD and NEW policies
prompt_old = BASE_PROMPT.format(policy=OLD_POLICY, item=food_item)
prompt_new = BASE_PROMPT.format(policy=NEW_POLICY, item=food_item)

# Get TrendBot's decision under each policy
label_old = normalize_label(ask_llm(prompt_old))
label_new = normalize_label(ask_llm(prompt_new))

# Show the results
print(f"Menu item:       {food_item}")
print(f"OLD policy says: {label_old}")
print(f"NEW policy says: {label_new}")
print()

if label_old == label_new:
    print("⚠ Both policies agree — try a different item to find a conflict!")
else:
    print("✓ The policies disagree — this item behaves differently under drift!")

Menu item:       Spicy fried chicken sandwich with extra cheese
OLD policy says: ADD
NEW policy says: ADD

⚠ Both policies agree — try a different item to find a conflict!


# Step 11 — Reflect: Did We Retrain?

Before you scroll down, try answering these yourself:

1. Did we change anything inside TrendBot — its weights, its memory, its training?
2. What exactly did we change to improve its accuracy from Step 7 to Step 8?
3. Can you think of a situation where *updating the prompt still wouldn't be enough?*

---

**Here's what happened:**
- We did **not** retrain TrendBot. We never touched the model itself.
- We only changed the **policy text** inside the prompt.
- That one change took accuracy from ~50% to 100%.

**But here's the limit:** if a model is too small or too rigid to follow policy rules
even when they're clearly written out, prompt updates won't save you.
That's when the real solution is **retraining** — teaching the model new behavior
from the ground up, not just telling it what to do.